# 02 - Scoring & Ranking Sanity Check

Before trusting the resume screening pipeline, we run it against real,
ID-pinned resumes from the bundled corpus and check that it behaves the way
a human recruiter would expect: candidates from a matching job category
should rank clearly above candidates from an unrelated category.

This is the same check encoded as an automated test in
`backend/tests/test_ranking_sanity.py` - here we run it interactively and
inspect *why* the ranking looks the way it does, using the real pipeline
modules (`app.preprocessing`, `app.skills`, `app.scoring`, `app.ranking`).

In [1]:
import sys
sys.path.insert(0, "../backend")

import pandas as pd

from app.datasets import get_resume_by_id
from app.ranking import RawCandidate, rank_candidates
from app.schemas import JobDescription

pd.set_option("display.max_colwidth", 60)

## Set up the job description and candidate pool

We use the curated "Information Technology Support Engineer" role from
`backend/app/data/curated_jobs.json`, and pick three real IT resumes and
three real Chef resumes (also used as the bundled sample PDFs) by their
Kaggle dataset ID.

In [2]:
IT_JOB = JobDescription(
    id="it-support-engineer",
    title="Information Technology Support Engineer",
    category="INFORMATION-TECHNOLOGY",
    description=(
        "We are looking for an IT Support Engineer to maintain our network "
        "infrastructure and provide help desk support to staff. You will manage "
        "user accounts in Active Directory, troubleshoot hardware and software "
        "issues, administer Windows Server environments, and write SQL queries "
        "to investigate data issues."
    ),
    required_skills=["Networking", "Troubleshooting", "Active Directory", "Help Desk Support", "Windows Server", "SQL"],
    preferred_skills=["Amazon Web Services (AWS)", "Docker", "Python", "Linux", "Cloud Computing"],
)

IT_RESUME_IDS = ["10089434", "10247517", "10265057"]
CHEF_RESUME_IDS = ["10001727", "10276858", "10333299"]

def load_candidates(resume_ids):
    candidates = []
    for resume_id in resume_ids:
        resume = get_resume_by_id(resume_id)
        candidates.append(
            RawCandidate(id=resume["id"], source="sample", text=resume["text"], category=resume["category"])
        )
    return candidates

candidates = load_candidates(IT_RESUME_IDS) + load_candidates(CHEF_RESUME_IDS)
[c.id for c in candidates]

['10089434', '10247517', '10265057', '10001727', '10276858', '10333299']

## Run the pipeline and inspect the ranking

In [3]:
results = rank_candidates(job=IT_JOB, candidates=candidates, skill_weight=0.6, similarity_weight=0.4)

table = pd.DataFrame([
    {
        "rank": r.rank,
        "id": r.id,
        "category": r.category,
        "skill_score": r.skill_score,
        "similarity_score": r.similarity_score,
        "final_score": r.final_score,
        "missing_required": ", ".join(r.missing_required_skills),
    }
    for r in results
]).sort_values("rank")

table

,rank,id,category,skill_score,similarity_score,final_score,missing_required
0,1,10089434,INFORMATION-TECHNOLOGY,0.8235,1.0000,0.8941,
1,2,10247517,INFORMATION-TECHNOLOGY,0.4706,0.6742,0.5520,"Active Directory, Help Desk Support"
2,3,10265057,INFORMATION-TECHNOLOGY,0.4118,0.3136,0.3725,"Active Directory, Help Desk Support, Windows Server"
3,4,10333299,CHEF,0.1176,0.0867,0.1053,"Troubleshooting, Active Directory, Help Desk Support, Wi..."
4,5,10001727,CHEF,0.0000,0.0789,0.0315,"Networking, Troubleshooting, Active Directory, Help Desk..."
5,6,10276858,CHEF,0.0000,0.0000,0.0000,"Networking, Troubleshooting, Active Directory, Help Desk..."


## Sanity check

Every Information Technology resume should rank above every Chef resume for
this IT job description - if that's not true, something is wrong with the
scoring before we ever ship it.

In [4]:
max_it_rank = table[table["category"] == "INFORMATION-TECHNOLOGY"]["rank"].max()
min_chef_rank = table[table["category"] == "CHEF"]["rank"].min()

print(f"Worst-ranked IT candidate: #{max_it_rank}")
print(f"Best-ranked Chef candidate: #{min_chef_rank}")
assert max_it_rank < min_chef_rank, "An IT-vs-CHEF ranking inversion was found!"
print("PASSED: every IT resume outranks every Chef resume for this IT job description.")

Worst-ranked IT candidate: #3
Best-ranked Chef candidate: #4
PASSED: every IT resume outranks every Chef resume for this IT job description.


## Reading one candidate's explanation

This is the same plain-English text a recruiter would see in the app's
candidate detail panel - generated purely from the numbers above, so it can
never contradict them.

In [5]:
top_candidate = results[0]
print(top_candidate.explanation)

Ranked #1 of 6. Matches 6 of 6 required skills and 2 of 5 preferred skills. The resume's overall content is a 100% textual match to the job description relative to the other candidates in this batch. Missing preferred skills: Amazon Web Services (AWS), Docker, Python.
